# Personal Finance EDA

Exploratory analysis of cleaned student finance transactions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'transactions_clean.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['Date'])
df.head()

In [ ]:
df.info()
df.describe(include='all')

## Monthly Spending Trend

In [ ]:
monthly_expense = (
    df[df['Income_Expense'] == 'Expense']
    .assign(Year_Month=lambda x: x['Date'].dt.to_period('M').astype(str))
    .groupby('Year_Month', as_index=False)['Amount']
    .sum()
)

px.line(monthly_expense, x='Year_Month', y='Amount', markers=True, title='Monthly Spending Trend')

## Income vs Expense

In [ ]:
monthly_type = (
    df.assign(Year_Month=lambda x: x['Date'].dt.to_period('M').astype(str))
    .groupby(['Year_Month', 'Income_Expense'], as_index=False)['Amount']
    .sum()
)

px.bar(monthly_type, x='Year_Month', y='Amount', color='Income_Expense', barmode='group', title='Income vs Expense by Month')

## Category Distribution

In [ ]:
category_spend = (
    df[df['Income_Expense'] == 'Expense']
    .groupby('Category', as_index=False)['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
)

px.pie(category_spend, names='Category', values='Amount', title='Expense Distribution by Category')

## Merchant Analysis

In [ ]:
top_merchants = (
    df[df['Income_Expense'] == 'Expense']
    .groupby('Merchant', as_index=False)['Amount']
    .sum()
    .sort_values('Amount', ascending=False)
    .head(10)
)

px.bar(top_merchants, x='Amount', y='Merchant', orientation='h', title='Top 10 Merchants by Spend')

## Payment Method Analysis

In [ ]:
payment_summary = (
    df.groupby(['Payment_Method', 'Income_Expense'], as_index=False)['Amount']
    .sum()
)

px.bar(payment_summary, x='Payment_Method', y='Amount', color='Income_Expense', barmode='group', title='Amount by Payment Method')

## Correlation Heatmap

In [ ]:
numeric_cols = ['Amount', 'Account_Balance', 'Year', 'Month', 'Is_Weekend']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## Monthly Savings

In [ ]:
monthly_pivot = monthly_type.pivot(index='Year_Month', columns='Income_Expense', values='Amount').fillna(0)
monthly_pivot['Net_Savings'] = monthly_pivot.get('Income', 0) - monthly_pivot.get('Expense', 0)
monthly_pivot = monthly_pivot.reset_index()

px.line(monthly_pivot, x='Year_Month', y='Net_Savings', markers=True, title='Monthly Net Savings')